In [1]:
import pandas as pd
import json
import subprocess
import os

In [2]:
stats = pd.read_csv("../data/disease_model_stats.csv")

## 0004992 - Generic Cancer

In [3]:
subprocess.run([
    "python", "../src/predict.py",
    "-input", "../demo/metadata_mouse_title_processed.tsv",
    "-id", "../demo/metadata_mouse_ID.tsv",
    "-input_embed", "../demo/metadata_mouse_title_embedding.npz",
    "-train_embed", "../data/disease_desc_embedding.npz",
    "-model", "../bins/MONDO_0004992__model.pkl",
    "-out", "../results/"
])

/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 1.35 s to load data
predicting labels
took 17.16 s to predict
retrieving predictive words
took 0.38 s to retrieve predictive words
saving output
took 0.32 min to load, predict, retrieve predictive words and save MONDO_0004992 for 4066 instances


CompletedProcess(args=['python', '../src/predict.py', '-input', '../demo/metadata_mouse_title_processed.tsv', '-id', '../demo/metadata_mouse_ID.tsv', '-input_embed', '../demo/metadata_mouse_title_embedding.npz', '-train_embed', '../data/disease_desc_embedding.npz', '-model', '../bins/MONDO_0004992__model.pkl', '-out', '../results/'], returncode=0)

In [4]:
preds_0004992 = pd.read_csv("../results/MONDO_0004992__preds.csv")
preds_0004992[preds_0004992['log2(prob/prior)'] > 0]

,ID,prob,log2(prob/prior),related_words
0,GSE87388,0.961283,0.487215,"cancer,pancreatic,tumor"
1,GSE81941,0.958960,0.483724,"breast,cancer,cell"
2,GSE74490,0.953467,0.475437,"cancer,pancreatic"
3,GSE59831,0.949153,0.468895,"analysis,cancer,cells,epithelial,lung,rnaseq,t..."
4,GSE77623,0.946461,0.464798,"cancer,expression,prostate,tumor"
...,...,...,...,...
90,GSE97489,0.692138,0.013315,"sarcoma,soft,tissues"
91,GSE66068,0.692047,0.013125,"acute,inhibition,leukemia,myeloid,rnaseq"
92,GSE67790,0.690774,0.010469,"cancer,colon,liver"
93,GSE74650,0.689315,0.007417,"acute,cells,inhibition,leukemia,myeloid,rnaseq..."


## Functions:

In [5]:
def load_children_with_models(path):
    with open(path) as f:
        children = json.load(f)
    terms = children['_embedded']['terms']
    ids = [t['short_form'] for t in terms]
    names = [t['label'] for t in terms]
    models = [
        stats['ID'].str.replace(":", "_").str.contains(id).any()
        for id in ids
    ]
    df = pd.DataFrame({'ids': ids, 'names': names, 'models': models})
    return df

def run_model_predictions(children_df, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for _, row in children_df.iterrows():
        if row['models']:
            subprocess.run([
                "python", "../src/predict.py",
                "-input", "../demo/metadata_mouse_title_processed.tsv",
                "-id", "../demo/metadata_mouse_ID.tsv",
                "-input_embed", "../demo/metadata_mouse_title_embedding.npz",
                "-train_embed", "../data/disease_desc_embedding.npz",
                "-model", f"../bins/{row['ids']}__model.pkl",
                "-out", output_dir
            ])
            
def collect_predictions(output_dir):
    files = [f for f in os.listdir(output_dir) if f.endswith(".csv")]
    dfs = []
    for f in files:
        df = pd.read_csv(os.path.join(output_dir, f))
        df['source'] = f.replace("__preds.csv", "")
        dfs.append(df)
    all_preds = pd.concat(dfs, ignore_index=True)
    all_preds.to_csv(os.path.join(output_dir, "all_cancer_predictions.csv"), index=False)
    print(f"Saved 'all_cancer_predictions.csv'")
    return all_preds

# Cancer Children

## Model Overview

In [7]:
children_cancer = load_children_with_models("../data/children_cancer.json")
children_cancer.to_csv("../data/children_cancer.csv", index=False)

## Prediction

In [15]:
run_model_predictions(children_cancer, "../results/mouse_children_cancer/")

/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 0.87 s to load data
predicting labels
took 10.50 s to predict
retrieving predictive words
took 0.18 s to retrieve predictive words
saving output
took 0.19 min to load, predict, retrieve predictive words and save MONDO_0024881 for 4066 instances


/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 0.85 s to load data
predicting labels
took 10.69 s to predict
retrieving predictive words
took 0.19 s to retrieve predictive words
saving output
took 0.20 min to load, predict, retrieve predictive words and save MONDO_0024637 for 4066 instances


/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


...

loading data
took 0.85 s to load data
predicting labels
took 13.92 s to predict
retrieving predictive words
took 0.23 s to retrieve predictive words
saving output
took 0.25 min to load, predict, retrieve predictive words and save MONDO_0000376 for 4066 instances


/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 0.94 s to load data
predicting labels
took 7.80 s to predict
retrieving predictive words
took 0.29 s to retrieve predictive words
saving output
took 0.15 min to load, predict, retrieve predictive words and save MONDO_0000637 for 4066 instances


/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 0.89 s to load data
predicting labels
took 8.31 s to predict
retrieving predictive words
took 0.35 s to retrieve predictive words
saving output
took 0.16 min to load, predict, retrieve predictive words and save MONDO_0000621 for 4066 instances


## Distribution

In [18]:
children_cancer_preds = collect_predictions("../results/mouse_children_cancer/")
children_cancer_preds[children_cancer_preds['log2(prob/prior)']>0].groupby('source').size().reset_index(name='count').sort_values('count', ascending=False)

Saved 'all_cancer_predictions.csv'


,source,count
1,MONDO_0000621,346
2,MONDO_0000637,259
21,MONDO_0020665,231
6,MONDO_0002516,230
24,MONDO_0024881,221
13,MONDO_0005872,208
9,MONDO_0004993,187
22,MONDO_0021069,151
23,MONDO_0024637,136
12,MONDO_0005853,135


# 0000621 (immune system cancer) Children

In [21]:
# ! wget https://www.ebi.ac.uk/ols4/api/ontologies/mondo/terms/http%253A%252F%252Fpurl.obolibrary.org%252Fobo%252FMONDO_0000621/children

## Model Overview

In [22]:
children_0000621 = load_children_with_models("../data/children_0000621.json")
children_0000621

,ids,names,models
0,MONDO_0021138,bone marrow cancer,True
1,MONDO_0019475,subcutaneous panniculitis-like T-cell lymphoma,False
2,MONDO_0019024,mast cell sarcoma,False
3,MONDO_0018223,systemic Epstein-Barr virus-positive T-cell ly...,False
4,MONDO_0015819,indolent primary cutaneous B-cell lymphoma,False
5,MONDO_0009693,plasma cell myeloma,True
6,MONDO_0006462,thyroid gland diffuse large B-cell lymphoma,False
7,MONDO_0006418,small intestinal enteropathy-associated T-cell...,False
8,MONDO_0006417,small intestinal diffuse large B-cell lymphoma,False
9,MONDO_0006416,small intestinal Burkitt lymphoma,False


## Prediction

In [23]:
run_model_predictions(children_0000621, "../results/mouse_children_0000621/")

/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 0.85 s to load data
predicting labels
took 12.17 s to predict
retrieving predictive words
took 0.34 s to retrieve predictive words
saving output
took 0.22 min to load, predict, retrieve predictive words and save MONDO_0021138 for 4066 instances


/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 0.84 s to load data
predicting labels
took 12.97 s to predict
retrieving predictive words
took 0.19 s to retrieve predictive words
saving output
took 0.23 min to load, predict, retrieve predictive words and save MONDO_0009693 for 4066 instances


/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 0.85 s to load data
predicting labels
took 12.07 s to predict
retrieving predictive words
took 0.16 s to retrieve predictive words
saving output
took 0.22 min to load, predict, retrieve predictive words and save MONDO_0000612 for 4066 instances


/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 0.85 s to load data
predicting labels
took 12.02 s to predict
retrieving predictive words
took 0.17 s to retrieve predictive words
saving output
took 0.22 min to load, predict, retrieve predictive words and save MONDO_0000872 for 4066 instances


/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 0.87 s to load data
predicting labels
took 12.01 s to predict
retrieving predictive words
took 0.17 s to retrieve predictive words
saving output
took 0.22 min to load, predict, retrieve predictive words and save MONDO_0000871 for 4066 instances


## Distribution

In [24]:
children_0000621_preds = collect_predictions("../results/mouse_children_0000621/")
children_0000621_preds[children_0000621_preds['log2(prob/prior)']>0].groupby('source').size().reset_index(name='count').sort_values('count', ascending=False)

Saved 'all_cancer_predictions.csv'


,source,count
4,MONDO_0021138,228
2,MONDO_0000872,116
1,MONDO_0000871,75
3,MONDO_0009693,56
0,MONDO_0000612,9


# 0021138 (bone marrow cancer) children

In [26]:
# ! wget https://www.ebi.ac.uk/ols4/api/ontologies/mondo/terms/http%253A%252F%252Fpurl.obolibrary.org%252Fobo%252FMONDO_0021138/children

## Model Overview

In [30]:
children_0021138 = load_children_with_models("../data/children_0021138.json")
children_0021138[children_0021138['models']]

,ids,names,models
0,MONDO_0020076,myeloproliferative neoplasm,True


## Prediction

In [32]:
run_model_predictions(children_0021138, "../results/mouse_children_0021138/")

/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 0.90 s to load data
predicting labels
took 7.03 s to predict
retrieving predictive words
took 0.32 s to retrieve predictive words
saving output
took 0.14 min to load, predict, retrieve predictive words and save MONDO_0020076 for 4066 instances


## Distribution

In [34]:
children_0021138_preds = collect_predictions("../results/mouse_children_0021138/")
children_0021138_preds[children_0021138_preds['log2(prob/prior)']>0].groupby('source').size().reset_index(name='count').sort_values('count', ascending=False)

Saved 'all_cancer_predictions.csv'


,source,count
0,MONDO_0020076,227


# 0020076 () Children

In [42]:
# ! wget https://www.ebi.ac.uk/ols4/api/ontologies/mondo/terms/http%253A%252F%252Fpurl.obolibrary.org%252Fobo%252FMONDO_0020076/children

In [43]:
children_0020076 = load_children_with_models("../data/children_0020076.json")
children_0020076[children_0020076['models']]

,ids,names,models
10,MONDO_0006311,myelodysplastic/myeloproliferative neoplasm,True
11,MONDO_0005029,essential thrombocythemia,True
12,MONDO_0004643,myeloid leukemia,True


In [39]:
run_model_predictions(children_0020076, "../results/mouse_children_0020076/")

/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 0.90 s to load data
predicting labels
took 10.17 s to predict
retrieving predictive words
took 0.16 s to retrieve predictive words
saving output
took 0.19 min to load, predict, retrieve predictive words and save MONDO_0006311 for 4066 instances


/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 0.86 s to load data
predicting labels
took 9.94 s to predict
retrieving predictive words
took 0.16 s to retrieve predictive words
saving output
took 0.18 min to load, predict, retrieve predictive words and save MONDO_0005029 for 4066 instances


/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 0.86 s to load data
predicting labels
took 10.00 s to predict
retrieving predictive words
took 0.33 s to retrieve predictive words
saving output
took 0.19 min to load, predict, retrieve predictive words and save MONDO_0004643 for 4066 instances


In [40]:
children_0020076_preds = collect_predictions("../results/mouse_children_0020076/")
children_0020076_preds[children_0020076_preds['log2(prob/prior)']>0].groupby('source').size().reset_index(name='count').sort_values('count', ascending=False)

Saved 'all_cancer_predictions.csv'


,source,count
0,MONDO_0004643,220
1,MONDO_0005029,4


# 0004643 () Children

In [45]:
# ! wget https://www.ebi.ac.uk/ols4/api/ontologies/mondo/terms/http%253A%252F%252Fpurl.obolibrary.org%252Fobo%252FMONDO_0004643/children

--2025-06-26 18:20:32--  https://www.ebi.ac.uk/ols4/api/ontologies/mondo/terms/http%253A%252F%252Fpurl.obolibrary.org%252Fobo%252FMONDO_0004643/children
Resolving www.ebi.ac.uk (www.ebi.ac.uk)... 193.62.193.80
Connecting to www.ebi.ac.uk (www.ebi.ac.uk)|193.62.193.80|:443... connected.
HTTP request sent, awaiting response... 200 
Length: unspecified [application/json]
Saving to: ‘children’

children                [ <=>                ]  52.92K  --.-KB/s    in 0.1s    

2025-06-26 18:20:33 (390 KB/s) - ‘children’ saved [54191]



In [46]:
children_0004643 = load_children_with_models("../data/children_0004643.json")
children_0004643[children_0004643['models']]

run_model_predictions(children_0004643, "../results/mouse_children_0004643/")

children_0004643_preds = collect_predictions("../results/mouse_children_0004643/")
children_0004643_preds[children_0004643_preds['log2(prob/prior)']>0].groupby('source').size().reset_index(name='count').sort_values('count', ascending=False)

/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 0.97 s to load data
predicting labels
took 11.40 s to predict
retrieving predictive words
took 0.31 s to retrieve predictive words
saving output
took 0.21 min to load, predict, retrieve predictive words and save MONDO_0011996 for 4066 instances


/home/rfu_smith_edu/.local/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


loading data
took 0.88 s to load data
predicting labels
took 13.08 s to predict
retrieving predictive words
took 0.21 s to retrieve predictive words
saving output
took 0.24 min to load, predict, retrieve predictive words and save MONDO_0018874 for 4066 instances
Saved 'all_cancer_predictions.csv'


,source,count
0,MONDO_0011996,1382
1,MONDO_0018874,122


# 0011996 () Children

In [50]:
# ! wget https://www.ebi.ac.uk/ols4/api/ontologies/mondo/terms/http%253A%252F%252Fpurl.obolibrary.org%252Fobo%252FMONDO_0011996/children

In [52]:
children_0011996 = load_children_with_models("../data/children_0011996.json")
children_0011996

,ids,names,models
0,MONDO_0021367,"leukemia, myeloid, accelerated-phase",False
1,MONDO_0010809,familial chronic myelocytic leukemia-like synd...,False
2,MONDO_0006115,"blast phase chronic myelogenous leukemia, BCR-...",False


In [53]:

# run_model_predictions(children_0011996, "../results/mouse_children_0011996/")

# children_0011996_preds = collect_predictions("../results/mouse_children_0011996/")
# children_0011996_preds[children_0011996_preds['log2(prob/prior)']>0].groupby('source').size().reset_index(name='count').sort_values('count', ascending=False)